# bigcode EDA

- BigCodeBench (ICLR 2025) 
- BigCodeBench-Instruct(자연어 지향 명령어를 기반으로 한 코드 생성) : **1,140 problems**
- 출처 : https://huggingface.co/datasets/bigcode/bigcodebench   
- 설명 : LLM 에이전트 , 즉 자연어 설명이나 코드 조각으로부터 프로그램을 생성할 수 있는 코드 생성 AI 시스템이 아닌 LLM 자체를 평가하는 데 있어 핵심적인 벤치마크 역할
- 평가지표 : pass@1

## overview :
- ch01 : 데이터 톺아보기
- ch02 : task 활용 관점으로 데이터 분석

---
## ch01 : 데이터 톺아보기   

01. 데이터 확인
02. 한 샘플 자세히 보기
03. **동작 흐름 관점 분석**
04. 연구 관점 정리

### 1. 데이터 확인하기

In [1]:
# 1. 데이터 확인
from pathlib import Path
import json
import pandas as pd

DATA_PATH = Path("../../datasets/bigcode/raw/bigcode.jsonl")

rows = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df.head()

,task_id,complete_prompt,instruct_prompt,canonical_solution,code_prompt,test,entry_point,doc_struct,libs
0,BigCodeBench/0,import itertools\nfrom random import shuffle\n...,Calculates the average of the sums of absolute...,permutations = list(itertools.permutations...,import itertools\nfrom random import shuffle\n...,import unittest\nfrom unittest.mock import pat...,task_func,"{""description"": [""Calculates the average of th...","['random', 'itertools']"
1,BigCodeBench/1,import collections\nimport random\nimport stri...,Generate a random string of the specified leng...,if length < 0:\n raise ValueError\n...,import collections\nimport random\nimport stri...,import unittest\nimport string\nclass TestCase...,task_func,"{""description"": [""Generate a random string of ...","['collections', 'random', 'string']"
2,BigCodeBench/2,import random\nimport statistics\n\ndef task_f...,Create a dictionary in which keys are random l...,"random_dict = {k: [random.randint(0, 100) ...",import random\nimport statistics\ndef task_fun...,import unittest\nclass TestCases(unittest.Test...,task_func,"{""description"": [""Create a dictionary in which...","['statistics', 'random']"
3,BigCodeBench/3,import random\nimport numpy as np\n\ndef task_...,Create a dictionary where keys are specified l...,"random_dict = {k: [random.randint(0, 100) ...",import random\nimport numpy as np\ndef task_fu...,import unittest\n \nclass TestCases(unittes...,task_func,"{""description"": [""Create a dictionary where ke...","['numpy', 'random']"
4,BigCodeBench/4,from collections import Counter\nimport iterto...,Count the occurrence of each integer in the va...,count_dict = Counter(itertools.chain.from_...,from collections import Counter\nimport iterto...,import unittest\nclass TestCases(unittest.Test...,task_func,"{""description"": [""Count the occurrence of each...","['collections', 'itertools']"


In [2]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())
df.dtypes

shape: (1140, 9)
columns: ['task_id', 'complete_prompt', 'instruct_prompt', 'canonical_solution', 'code_prompt', 'test', 'entry_point', 'doc_struct', 'libs']


task_id               object
complete_prompt       object
instruct_prompt       object
canonical_solution    object
code_prompt           object
test                  object
entry_point           object
doc_struct            object
libs                  object
dtype: object

### 1-2. 데이터 필드 확인하기   
출처 : https://huggingface.co/datasets/bigcode/bigcodebench

**Data Fields:**   
- task_id( string): 작업에 대한 고유 식별자
- complete_prompt( string): PEP257 구조의 문서 문자열 프롬프트
- **instruct_prompt( string): 자연어 중심의 지시문**
- canonical_solution( string): 주석이 없는 표준 해법.
- code_prompt( string): 코드만 입력하라는 메시지
- test( string): 테스트용 코드 조각을 unittest.TestCase클래스로 묶은 것
- entry_point( string): 코드 스니펫의 진입점 task_func.
- doc_struct( string[dictionary]): 구조화된 문서 문자열
- description( string): 자연어로 작성된 주요 작업 설명.
- note( string): 작업에 대한 추가 설명은 자연어로 작성
- reqs( string, optional): 해당 모듈은 과제 해결에 사용할 수 있음
- params( string, optional): 과제 해결에 사용된 매개변수
- returns( string, optional): 작업 결과에서 반환될 값
- raises( string, optional): 예외는 작업 해결 과정에서 발생해야 함
- examples( string, optional): 과제 해결에 대한 힌트로 제공되는 대화형 Python 예제
- libs( string): 해당 라이브러리는 과제 해결에 사용할 수 있음

### 2. 샘플 데이터 자세히 뜯어보기

In [3]:
# 샘플 1개에 대해 확인하기
sample = df.iloc[0].to_dict()
for k, v in sample.items():
    print(f"\n===== {k} =====")
    print(v if isinstance(v, str) else repr(v))


===== task_id =====
BigCodeBench/0

===== complete_prompt =====
import itertools
from random import shuffle

def task_func(numbers=list(range(1, 3))):
    """
    Calculates the average of the sums of absolute differences between each pair of consecutive numbers 
    for all permutations of a given list. Each permutation is shuffled before calculating the differences.

    Args:
    - numbers (list): A list of numbers. Default is numbers from 1 to 10.
    
    Returns:
    float: The average of the sums of absolute differences for each shuffled permutation of the list.

    Requirements:
    - itertools
    - random.shuffle

    Example:
    >>> result = task_func([1, 2, 3])
    >>> isinstance(result, float)
    True
    """


===== instruct_prompt =====
Calculates the average of the sums of absolute differences between each pair of consecutive numbers for all permutations of a given list. Each permutation is shuffled before calculating the differences. Args: - numbers (list): A list 

### 3. 동작 흐름 관점 분석

한 샘플의 동작 :    
(1) input : text  

(2) output code : 모델이 함수 전체 생성  
    → code = 함수 이름, 파라미터, 클래스, 로직을 다 생성해야함  

(3) exec(code)  
    → 함수 생성

(4) exec(test)  
    → check 함수 생성(import 등 환경 설정)  

(5) check(모델이 만든 함수)를 실행하면 예제들 넣어서 정답확인


(1) input : text   
**모델 입력값 :  자연어 입력** 

In [ ]:
# ===== instruct_prompt =====
# Calculates the average of the sums of absolute differences between each pair of consecutive numbers for all permutations of a given list. Each permutation is shuffled before calculating the differences. Args: - numbers (list): A list of numbers. Default is numbers from 1 to 10.
# The function should output with:
#     float: The average of the sums of absolute differences for each shuffled permutation of the list.
# You should write self-contained code starting with:
# ```
# import itertools
# from random import shuffle
# def task_func(numbers=list(range(1, 3))):
# ```

### 4. 연구 포인트

자연어와 코드 조각을 입력함

---
## ch02 : 데이터 톺아보기   

01. 전체 길이 분석
02. input/output 관점
03. Test 구조 분석
04. Research implication 

### 1. 길이 관점

In [7]:
# text_len : 모델에 들어가는 길이
# code_len : 모델이 출력하는 길이 관점 추정
# test_list_len : 평가 복잡도? 검증 조건의 길이

# 길이 (문자 기준)
for col in ["instruct_prompt", "canonical_solution", "test"]:
    df[f"{col}_len"] = df[col].astype(str).str.len()

# 개수 (리스트 기준)
df["instruct_prompt_count"] = df["instruct_prompt"].apply(len)
df["canonical_solution_count"] = df["canonical_solution"].apply(len)
df["test_count"] = df["test"].apply(len)

# 확인
df[[
    "instruct_prompt_len",
    "canonical_solution_len",
    "test_len",
    "instruct_prompt_count",
    'canonical_solution_count',
    "test_count"
]].describe()

,instruct_prompt_len,canonical_solution_len,test_len,instruct_prompt_count,canonical_solution_count,test_count
count,1140.000000,1140.000000,1140.00000,1140.000000,1140.000000,1140.00000
mean,662.742982,466.214035,2360.02807,662.742982,466.214035,2360.02807
std,277.653323,250.179872,1187.58841,277.653323,250.179872,1187.58841
min,232.000000,78.000000,568.00000,232.000000,78.000000,568.00000
25%,472.000000,279.750000,1591.00000,472.000000,279.750000,1591.00000
50%,606.500000,414.000000,2116.50000,606.500000,414.000000,2116.50000
75%,776.250000,590.500000,2874.00000,776.250000,590.500000,2874.00000
max,3464.000000,1811.000000,14791.00000,3464.000000,1811.000000,14791.00000


In [8]:
preview_cols = ['task_id', 'instruct_prompt', 'canonical_solution', 'test', ]
df[preview_cols].head(5)

,task_id,instruct_prompt,canonical_solution,test
0,BigCodeBench/0,Calculates the average of the sums of absolute...,permutations = list(itertools.permutations...,import unittest\nfrom unittest.mock import pat...
1,BigCodeBench/1,Generate a random string of the specified leng...,if length < 0:\n raise ValueError\n...,import unittest\nimport string\nclass TestCase...
2,BigCodeBench/2,Create a dictionary in which keys are random l...,"random_dict = {k: [random.randint(0, 100) ...",import unittest\nclass TestCases(unittest.Test...
3,BigCodeBench/3,Create a dictionary where keys are specified l...,"random_dict = {k: [random.randint(0, 100) ...",import unittest\n \nclass TestCases(unittes...
4,BigCodeBench/4,Count the occurrence of each integer in the va...,count_dict = Counter(itertools.chain.from_...,import unittest\nclass TestCases(unittest.Test...


### 2. intput/output 구조 관찰

In [9]:
for i in range(3):
    print(f"\n================ SAMPLE {i} ================ len : {len(df.loc[i, 'instruct_prompt'])}\n")
    print(df.loc[i, "instruct_prompt"])


================ SAMPLE 0 ================ len : 559

Calculates the average of the sums of absolute differences between each pair of consecutive numbers for all permutations of a given list. Each permutation is shuffled before calculating the differences. Args: - numbers (list): A list of numbers. Default is numbers from 1 to 10.
The function should output with:
    float: The average of the sums of absolute differences for each shuffled permutation of the list.
You should write self-contained code starting with:
```
import itertools
from random import shuffle
def task_func(numbers=list(range(1, 3))):
```

================ SAMPLE 1 ================ len : 581

Generate a random string of the specified length composed of uppercase and lowercase letters, and then count the occurrence of each character in this string.
The function should raise the exception for: ValueError if the length is a negative number
The function should output with:
    dict: A dictionary where each key is a chara

In [10]:
for i in range(3):
    print(f"\n================ SAMPLE {i} ================ len : {len(df.loc[i, 'canonical_solution'])}\n")
    print(df.loc[i, "canonical_solution"])


================ SAMPLE 0 ================ len : 337

    permutations = list(itertools.permutations(numbers))
    sum_diffs = 0

    for perm in permutations:
        perm = list(perm)
        shuffle(perm)
        diffs = [abs(perm[i] - perm[i+1]) for i in range(len(perm)-1)]
        sum_diffs += sum(diffs)

    avg_sum_diffs = sum_diffs / len(permutations)
    
    return avg_sum_diffs

================ SAMPLE 1 ================ len : 228

    if length < 0:
        raise ValueError
    random_string = ''.join(random.choices(string.ascii_uppercase + string.ascii_lowercase, k=length))
    char_counts = collections.Counter(random_string)
    return dict(char_counts)

================ SAMPLE 2 ================ len : 234

    random_dict = {k: [random.randint(0, 100) for _ in range(random.randint(1, 10))] for k in LETTERS}
    sorted_dict = dict(sorted(random_dict.items(), key=lambda item: statistics.mean(item[1]), reverse=True))
    return sorted_dict


### 3. test 구조 관찰

In [11]:
for i in range(3):
    print(f"\n================ TEST {i} - {df.loc[i, 'task_id']} ================ len : {len(df.loc[i, 'test'])}\n")
    print(df.loc[i, "test"])


================ TEST 0 - BigCodeBench/0 ================ len : 3062

import unittest
from unittest.mock import patch
from random import seed, shuffle
import itertools
class TestCases(unittest.TestCase):
    def test_default_numbers(self):
        # Test with default number range (1 to 10) to check that the result is a positive float.
        result = task_func()
        self.assertIsInstance(result, float)
        self.assertGreater(result, 0)
    def test_custom_list(self):
        # Test with a custom list of small positive integers to ensure proper handling and positive result.
        result = task_func([1, 2, 3])
        self.assertIsInstance(result, float)
        self.assertGreater(result, 0)
    def test_negative_numbers(self):
        # Test with negative numbers to verify the function handles and returns a positive result.
        result = task_func([-3, -2, -1])
        self.assertIsInstance(result, float)
        self.assertGreater(result, 0)
    def test_single_element(s

### 4. 연구관점 정리

output 길이가 78~1811 사이라서 max사이즈 수정 필요